In [17]:
# This generates langgraph code from graph_spec
from graph_gen.gen_graph import gen_graph

from langgraph.graph import StateGraph, START, END
from typing import Annotated

In [18]:
graph_spec = """
start(TheoryOfMind)
  => user_actual

user_actual
  => vom_toe_thought

vom_toe_thought
  => bot_response

bot_response
  => user_prediction, user_actual

user_prediction
  => revised_user_prediction

revised_user_prediction
  => voe_thought

voe_thought
  => derive_facts

derive_facts
  => store_facts
"""

graph_code = gen_graph("tom", graph_spec)
print(graph_code)

tom = StateGraph(TheoryOfMind)
tom.add_node('start', start)
tom.add_node('user_actual', user_actual)
tom.add_node('vom_toe_thought', vom_toe_thought)
tom.add_node('bot_response', bot_response)
tom.add_node('user_prediction', user_prediction)
tom.add_node('revised_user_prediction', revised_user_prediction)
tom.add_node('voe_thought', voe_thought)
tom.add_node('derive_facts', derive_facts)

tom.set_entry_point('start')

tom.add_edge('start', 'user_actual')
tom.add_edge('user_actual', 'vom_toe_thought')
tom.add_edge('vom_toe_thought', 'bot_response')
tom.add_edge('bot_response', 'user_prediction')
tom.add_edge('bot_response', 'user_actual')
tom.add_edge('user_prediction', 'revised_user_prediction')
tom.add_edge('revised_user_prediction', 'voe_thought')
tom.add_edge('voe_thought', 'derive_facts')
tom.add_edge('derive_facts', 'store_facts')

tom = tom.compile()


In [22]:
from langchain_anthropic import ChatAnthropic
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

class TheoryOfMind(StateGraph):
    user_input: str
    user_done: bool
    bot_response: str
    expected_user_to_say: str
    user_input_revised: str
    user_voe_thought: str
    user_voe_facts: list
    user_voe_reasons: list
    thought_for_bot: str
    messages: Annotated[list, add_messages]
    
def start(state):
    messages = [
        SystemMessage(content="You are a helpful assistant named Bob."),
    ]
    return { "user_input": None, "message": messages, "expected_user_to_say": "Nothing", "voe_reasons": ["no user input so far"] }

def user_actual(state):
    response = input("User: ")
    user_done = response.lower() in ["q", "quit", "bye", "x", "exit"]
    return { "user_input": response, "user_done": user_done }

prompt_vom_toe_thought = ChatPromptTemplate.from_template(
    """Given our previous conversation:
    {previous_conversation}

    We expected you to say: {expected_say}
    But you said: {actual_say}
    We were wrong because: {list_of_reasons}

    However, we now have a better understanding of how to continue
    the conversation, given you said:
    {actual_say}
    """)


llm_voe_toe_thought = ChatAnthropic(model="claude-3-5-sonnet-20240620")

def as_text(message):
    if isinstance(message, HumanMessage):
        return(f"User: {message.content}")
    elif isinstance(message, AIMessage):
        return(f"Assistant: {message.content}")
    else:
        return("")
        
def summarize(messages):
    result = []
    for message in messages:
        text = as_text(message)
        if text:
            result.append(text)
    return "\n".join(result)
    
def vom_toe_thought(state):
    previous_conversation = summarize(state["messages"])
    expected_say = state["expected_user_to_say"]
    actual_say = state["user_input"]
    list_of_reasons = "\n".join(state["user_voe_reasons"])
    response = llm_voe_toe_thought.invoke(
        {
            "previous_conversation": previous_conversation,
            "expected_say": expected_say,
            "actual_say": actual_say,
            "list_of_reasons": list_of_reasons,
        })
    return { "messages": [response]}

In [23]:
exec(graph_code)

NameError: name 'bot_response' is not defined

In [4]:
from IPython.display import Image, display

# Setting xray to 1 will show the internal structure of the nested graph
display(Image(tom.get_graph(xray=1).draw_mermaid_png()))

NameError: name 'tom' is not defined